In [ ]:
import torch
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def laplacian(grid):
    return (
        torch.roll(grid, 1, 0) +
        torch.roll(grid, -1, 0) +
        torch.roll(grid, 1, 1) +
        torch.roll(grid, -1, 1) -
        4 * grid
    )

def update(A, B, DA, DB, f, k, delta_t):
    diff_A = DA * laplacian(A)
    diff_B = DB * laplacian(B)


    reaction = A * (B ** 2)
    A += (diff_A - reaction + f * (1 - A)) * delta_t
    B += (diff_B + reaction - (k + f) * B) * delta_t
    return A, B


def get_initial_A_and_B(N, random_influence=0.2, device='cuda'):

    A = (1 - random_influence) * torch.ones((N, N), device=device)
    B = torch.zeros((N, N), device=device)

    A += random_influence * torch.rand((N, N), device=device)
    B += random_influence * torch.rand((N, N), device=device)

    N2, r = N // 2, N // 10
    A[N2-r:N2+r, N2-r:N2+r] = 0.50
    B[N2-r:N2+r, N2-r:N2+r] = 0.25

    return A, B

grid_sizes = [200,400,600, 800, 1000,1200, 1600]
n_steps = 1000

timings = []
for N in grid_sizes:

    A, B = get_initial_A_and_B(N, device=device)


    if device.type == 'cuda': torch.cuda.synchronize()
    start_time = time.time()

    for _ in range(n_steps):
        A, B = update(A, B, 0.16, 0.08, 0.060, 0.062, 1.0)


    if device.type == 'cuda': torch.cuda.synchronize()
    end_time = time.time()

    total_time = end_time - start_time
    timings.append(total_time)
    print(f"N={N}, grid={N}x{N}, steps={N_steps}, time={total_time:.3f}s")
    print(f"Grid: {N}x{N} | {n_steps} steps | Time: {:.4f}s")
print(timings)

In [ ]:
import numpy as np
import torch

# cpu original
def laplacian_numpy(mat):
    neigh_mat = -4 * mat.copy()
    neighbors = [(1.0, (-1, 0)), (1.0, (0, -1)), (1.0, (0, 1)), (1.0, (1, 0))]
    for weight, neigh in neighbors:
        neigh_mat += weight * np.roll(mat, neigh, (0, 1))
    return neigh_mat


def update_original(A, B, DA, DB, f, k, delta_t):
    diff_A = DA * laplacian_numpy(A)
    diff_B = DB * laplacian_numpy(B)

    reaction = A * B**2
    diff_A -= reaction
    diff_B += reaction

    diff_A += f * (1 - A)
    diff_B -= (k + f) * B

    A += diff_A * delta_t
    B += diff_B * delta_t
    return A, B


N_val = 200
steps_val = 100
torch.manual_seed(42)

A_gpu, B_gpu = get_initial_A_and_B(N_val, device=device)
A_cpu = A_gpu.cpu().numpy().copy()
B_cpu = B_gpu.cpu().numpy().copy()

for _ in range(steps_val):
    A_gpu, B_gpu = update(A_gpu, B_gpu, 0.16, 0.08, 0.060, 0.062, 1.0)

for _ in range(steps_val):
    A_cpu, B_cpu = update_original(A_cpu, B_cpu, 0.16, 0.08, 0.060, 0.062, 1.0)

A_gpu_final = A_gpu.cpu().numpy()
B_gpu_final = B_gpu.cpu().numpy()

print(f"Validation on Device: {device}")
print(f"A match: {np.allclose(A_cpu, A_gpu_final, atol=1e-6)}")
print(f"B match: {np.allclose(B_cpu, B_gpu_final, atol=1e-6)}")
print(f"Max Diff A: {np.max(np.abs(A_cpu - A_gpu_final)):.2e}")